### Libaraies

In [1]:
import numpy as np
import pandas as pd
import joblib
import onnx
import onnxruntime as rt
import os
import sys

import warnings
warnings.filterwarnings('ignore')

sys.path.append('../src')
from model_selection import load_final_models, FINAL_MODELS

### Step 1: Setup

In [5]:
# Paths
BASE_DIR    = '..'
MODEL_DIR   = f'{BASE_DIR}/models/Final_Models'
ONNX_DIR    = f'{BASE_DIR}/models_onnx'
PARAM_DIR   = f'{BASE_DIR}/models/Project_Parameter_Files'
DATA_DIR    = f'{BASE_DIR}/data/engineered_data'

os.makedirs(ONNX_DIR, exist_ok=True)

In [6]:
# Load final models
print("Loading final models...")
final_models = load_final_models(MODEL_DIR)

cls_model     = final_models['classification']
reg_model     = final_models['regression']
cluster_model = final_models['clustering']

# Load sample data for conversion
X_val_cls = pd.read_csv(f'{DATA_DIR}/X_val_cls_engineered.csv')
X_val_reg = pd.read_csv(f'{DATA_DIR}/X_val_reg_engineered.csv')

print(f"\nModels loaded")
print(f"Data loaded")
print(f"   X_val_cls : {X_val_cls.shape}")
print(f"   X_val_reg : {X_val_reg.shape}")
print(f"\nONNX output dir: {ONNX_DIR}")

Loading final models...
Loaded cls_LightGBM
Loaded reg_RandomForest
Loaded cluster_GMM

Models loaded
Data loaded
   X_val_cls : (2444, 23)
   X_val_reg : (2210, 16)

ONNX output dir: ../models_onnx


### Step 2: Convert LightGBM to ONNX

In [8]:
from onnxmltools import convert_lightgbm
from onnxmltools.convert.common.data_types import FloatTensorType

# Define input shape
n_features_cls = X_val_cls.shape[1]
initial_type   = [('float_input',FloatTensorType([None, n_features_cls]))]

# Convert
print("Converting LightGBM → ONNX...")
lgbm_onnx = convert_lightgbm(
    cls_model,
    initial_types=initial_type,
    target_opset=12)

# Save
onnx_path = f'{ONNX_DIR}/cls_LightGBM.onnx'
with open(onnx_path, 'wb') as f:
    f.write(lgbm_onnx.SerializeToString())

print(f" LightGBM ONNX saved: {onnx_path}")

# Verify file
size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"   File size: {size_mb:.2f} MB")

Converting LightGBM → ONNX...
 LightGBM ONNX saved: ../models_onnx/cls_LightGBM.onnx
   File size: 1.05 MB


### Step 3: Convert RandomForest to ONNX

In [9]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Define input shape
n_features_reg = X_val_reg.shape[1]
initial_type   = [('float_input',FloatTensorType([None, n_features_reg]))]

# Convert
print("Converting RandomForest → ONNX...")
rf_onnx = convert_sklearn(
    reg_model,
    initial_types=initial_type,
    target_opset=12)

# Save
onnx_path = f'{ONNX_DIR}/reg_RandomForest.onnx'
with open(onnx_path, 'wb') as f:
    f.write(rf_onnx.SerializeToString())

print(f" RandomForest ONNX saved: {onnx_path}")

# Verify file
size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"   File size: {size_mb:.2f} MB")

Converting RandomForest → ONNX...
 RandomForest ONNX saved: ../models_onnx/reg_RandomForest.onnx
   File size: 20.85 MB


###  Step 4: Convert GMM to ONNX

In [10]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Define input shape
n_features_cluster = X_val_cls.shape[1]
initial_type       = [('float_input',FloatTensorType([None, n_features_cluster]))]

# Convert
print("Converting GMM → ONNX...")
gmm_onnx = convert_sklearn(
    cluster_model,
    initial_types=initial_type,
    target_opset=12)

# Save
onnx_path = f'{ONNX_DIR}/cluster_GMM.onnx'
with open(onnx_path, 'wb') as f:
    f.write(gmm_onnx.SerializeToString())

print(f" GMM ONNX saved: {onnx_path}")

# Verify file
size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"   File size: {size_mb:.2f} MB")

Converting GMM → ONNX...
 GMM ONNX saved: ../models_onnx/cluster_GMM.onnx
   File size: 0.00 MB


In [ ]:
onnx_path = f'{ONNX_DIR}/cluster_GMM.onnx'
size_bytes = os.path.getsize(onnx_path)
print(f"File size: {size_bytes} bytes")

# Load and check
gmm_model_check = onnx.load(onnx_path)
print(f"ONNX IR version : {gmm_model_check.ir_version}")
print(f"Graph inputs    : {[i.name for i in gmm_model_check.graph.input]}")
print(f"Graph outputs   : {[o.name for o in gmm_model_check.graph.output]}")
print(f"Nodes in graph  : {len(gmm_model_check.graph.node)}")

File size: 1341 bytes
ONNX IR version : 7
Graph inputs    : ['float_input']
Graph outputs   : ['label', 'probabilities']
Nodes in graph  : 13


In [12]:
# 0.00 MB was just rounding — actual size is 1341 bytes = 1.3 KB

### Step 5: Verify all ONNX models — test inference

In [21]:
# Sample inputs
X_cls_sample = X_val_cls.iloc[:5].values.astype(np.float32)
X_reg_sample = X_val_reg.iloc[:5].values.astype(np.float32)

print("ONNX INFERENCE VERIFICATION")

# --- Classification ---
print("\n--- cls_LightGBM ---")
sess_cls   = rt.InferenceSession(f'{ONNX_DIR}/cls_LightGBM.onnx')
input_name = sess_cls.get_inputs()[0].name
outputs    = sess_cls.run(None, {input_name: X_cls_sample})
# Fix — LightGBM ONNX returns list of dicts
onnx_prob  = np.array([p[1] for p in outputs[1]])
print(f"Output names  : {[o.name for o in sess_cls.get_outputs()]}")
print(f"Labels        : {outputs[0]}")
print(f"Probabilities : {onnx_prob.round(4)}")

# --- Regression ---
print("\n--- reg_RandomForest ---")
sess_reg   = rt.InferenceSession(f'{ONNX_DIR}/reg_RandomForest.onnx')
input_name = sess_reg.get_inputs()[0].name
outputs    = sess_reg.run(None, {input_name: X_reg_sample})
print(f"Output names  : {[o.name for o in sess_reg.get_outputs()]}")
print(f"Predictions   : {outputs[0].round(4)}")

# --- Clustering ---
print("\n--- cluster_GMM ---")
sess_gmm   = rt.InferenceSession(f'{ONNX_DIR}/cluster_GMM.onnx')
input_name = sess_gmm.get_inputs()[0].name
outputs    = sess_gmm.run(None, {input_name: X_cls_sample})
print(f"Output names  : {[o.name for o in sess_gmm.get_outputs()]}")
print(f"Labels        : {outputs[0]}")
print(f"Probabilities : {outputs[1].round(4)}")

print("\n All ONNX models verified")

ONNX INFERENCE VERIFICATION

--- cls_LightGBM ---
Output names  : ['label', 'probabilities']
Labels        : [0 0 0 0 0]
Probabilities : [0.0983 0.0567 0.123  0.     0.0048]

--- reg_RandomForest ---
Output names  : ['variable']
Predictions   : [[17.4874]
 [ 7.3839]
 [ 7.4486]
 [16.6844]
 [11.2476]]

--- cluster_GMM ---
Output names  : ['label', 'probabilities']
Labels        : [[1]
 [1]
 [1]
 [1]
 [1]]
Probabilities : [[0.     1.    ]
 [0.     1.    ]
 [0.     1.    ]
 [0.3977 0.6023]
 [0.     1.    ]]

 All ONNX models verified


### Step 6: Comparison Orignal vs Predicted

In [25]:
print("ONNX vs ORIGINAL — MATCH CHECK")

BEST_THRESHOLD_CLS = 0.6

# --- Classification ---
print("\n--- cls_LightGBM ---")
orig_prob  = cls_model.predict_proba(X_val_cls.iloc[:5])[:, 1]
orig_pred  = (orig_prob >= BEST_THRESHOLD_CLS).astype(int)
onnx_out   = sess_cls.run(
    None, {'float_input': X_cls_sample})
onnx_prob  = np.array([p[1] for p in onnx_out[1]])
onnx_pred  = (onnx_prob >= BEST_THRESHOLD_CLS).astype(int)
print(f"Original probs : {orig_prob.round(4)}")
print(f"ONNX probs     : {onnx_prob.round(4)}")
print(f"Preds match    : {np.array_equal(orig_pred, onnx_pred)}")

# --- Regression ---
print("\n--- reg_RandomForest ---")
orig_reg  = reg_model.predict(X_val_reg.iloc[:5])
onnx_reg  = sess_reg.run(
    None, {'float_input': X_reg_sample})[0].ravel()
print(f"Original : {orig_reg.round(4)}")
print(f"ONNX     : {onnx_reg.round(4)}")
print(f"Match    : {np.allclose(orig_reg, onnx_reg, atol=1e-3)}")

# --- Clustering ---
print("\n--- cluster_GMM ---")
orig_labels = cluster_model.predict(X_val_cls.iloc[:5])
onnx_labels = sess_gmm.run(
    None, {'float_input': X_cls_sample})[0].ravel()
print(f"Original : {orig_labels}")
print(f"ONNX     : {onnx_labels}")
print(f"Match    : {np.array_equal(orig_labels, onnx_labels)}")

print("\nMatch check complete")

ONNX vs ORIGINAL — MATCH CHECK

--- cls_LightGBM ---
Original probs : [0.0983 0.0567 0.123  0.     0.0048]
ONNX probs     : [0.0983 0.0567 0.123  0.     0.0048]
Preds match    : True

--- reg_RandomForest ---
Original : [17.4874  7.3839  7.4486 16.6844 11.2475]
ONNX     : [17.4874  7.3839  7.4486 16.6844 11.2476]
Match    : True

--- cluster_GMM ---
Original : [1 1 1 1 1]
ONNX     : [1 1 1 1 1]
Match    : True

Match check complete


### Step 7: Document Report

In [26]:
# see report in docs/onnx_conversion_report.md